# Retrieval Validation

Validasi konfigurasi retrieval terbaik setelah scope guard. Notebook ini membangun satu index saja dan tidak mengulang sweep kalibrasi.

In [ ]:
import torch

assert torch.cuda.is_available(), "Pilih Runtime > Change runtime type > GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
BRANCH = "main"

%cd /content
!test -d indonesian-legal-compliance-rag || git clone --depth 1 --branch {BRANCH} https://github.com/FadhilahAfif/indonesian-legal-compliance-rag.git
!git -C indonesian-legal-compliance-rag checkout {BRANCH}
!git -C indonesian-legal-compliance-rag pull --ff-only origin {BRANCH}
%cd /content/indonesian-legal-compliance-rag

In [ ]:
!python -m pip install -q -r requirements.txt
!python -m pip uninstall -y -q torchvision torchcodec
!python scripts/check_environment.py
!python -m unittest discover -s tests -v
!python -m eval.validate_cases --require-reviewed

In [ ]:
!python -m gdown --folder https://drive.google.com/drive/folders/1LHZ1IncPmmUN5kytFu3i7MoaafFrKDql -O data/raw

In [ ]:
!python -m src.rag --device cuda --methods hybrid_rerank --parent-size 1000 --parent-overlap 100 --child-size 300 --child-overlap 30 --candidate-k 10 --bm25-weight 0.4 --threshold 0.3 --output-dir eval/results/final-retrieval

In [ ]:
import json
from pathlib import Path

report = json.loads(Path("eval/results/final-retrieval/retrieval_ablation.json").read_text())
report["results"]["hybrid_rerank"]

In [ ]:
from google.colab import files

files.download("eval/results/final-retrieval/retrieval_ablation.json")
files.download("eval/results/final-retrieval/retrieval_predictions.jsonl")